<a href="https://colab.research.google.com/github/pavankumarcode/Mastering-AI/blob/main/2__Agent__CRM_Lead_Qualifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CRM Lead Qualifier Agent

## Goal

To develop an AI agent that "automatically" enriches a new sales lead (identified by an email address) by gathering publicly available company information, checking for prior engagement in the internal CRM, and assigning a preliminary qualification score.

## Context

Sales representatives often spend valuable time manually researching leads and cross-referencing internal systems before a discovery call. This process is slow, inconsistent, and often leads to a poorly prepared first interaction.

## Agent Functionality

The agent must be able to:

1. Extract Domain: Take the email address and extract the company domain name (e.g., jane@acmecorp.com → acmecorp.com).

2. Enrich Company Data: Use the domain to look up (simulated) company details like industry, size, and annual revenue.

3. Check CRM History: Search the internal (simulated) CRM for any past contact or notes associated with the lead's email.

4. Calculate Lead Score: Synthesize all gathered data to assign a qualitative priority score (e.g., High, Medium, Low).

5. Final Summary: Present a concise, actionable summary of all findings to the sales representative.

# Initialize the Agent - Get all Imports

In [1]:
import os
import json
from openai import OpenAI
from google.colab import userdata

# Set up the connection to OpenAI

In [2]:
# 1. Initialize OpenAI Client
try:
    client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
except Exception as e:
    print(f"Error initializing OpenAI client: [{e}]")
    print("Please ensure your OPENAI_API_KEY is set in your environment variables.")

# Function to get Domain info for a specific Domain

In [6]:
"""
Based on the Domain received, gather all the information available
and return it back to the caller in JSON format.

For now, we have provided dummy data, in actual application this is where
we can connect to DB/Interet or other source to gather information.

"""

def get_domain_info(domain: str) -> str:

    print(f"Tool Called - get_domain_info : Looking up domain info for [{domain}]")

    # Mock database for testing, in real this is where we connect to other DB/Internet to get info.
    dummy_data = {
        "google.com" : {"industry": "Software" , "size": "501-1000 employees", "revenue": "$50M - $100M"},
        "amazon.com" : {"industry": "Commerce" , "size": "100-250 employees",  "revenue": "$10M - $25M"},
        "apple.com"  : {"industry": "Creaivity", "size": "5000+ employees",    "revenue": "$1B+"},
    }

    info = dummy_data.get(domain, {"industry": "Unknown", "size": "N/A", "revenue": "N/A"})

    # Return the data as a JSON string for the AI model to parse easily
    return json.dumps(info)

# Function to get the User info

In [5]:
"""
Gather all the information about the user and sent it back in JSON format.

For now, we have provided dummy data, in actual application this is where
we can connect to DB/Interet or other source to gather details about the user

In production system, this could be PostgreSQL other API, Salesforce etc.
"""
def get_user_info(email: str) -> str:

    print(f"Tool Called - get_user_info : Looking up User info for emailid [{email}]")

    # Mock database for demonstration
    dummy_data = {
        "jane@acmecorp.com": {"last_contact": "2025-11-15", "status": "Cold Lead",          "notes": "Attended webinar, no follow-up yet."},
        "bob@widgetco.net" : {"last_contact": "2025-12-01", "status": "Active Opportunity", "notes": "Discussed Q1 budget and product integration."},
        "default"          : {"last_contact": "N/A",        "status": "No Record",          "notes": "New lead, first contact opportunity."},
    }

    user_info = dummy_data.get(email, dummy_data["default"])
    return json.dumps(user_info)

# Function to calculate the Lead Score

In [ ]:
"""
Calculate the Lead Score based on Domain Informaiton and User data
And classify it to High, Medium or Low Category.

"""
def calculate_lead_score(data_summary: str) -> str:

    print(f"Tool Called - calculate_lead_score : Calculating the Lead Score for User [{data_summary}]")

    data = json.loads(data_summary)
    score = "Low" # Default score

    # Simple scoring logic for demonstration
    if data["domain_info"].get("revenue", "").startswith("$1B+"):
        score = "High"
    elif data["crm_history"].get("status") == "Active Opportunity":
        score = "High"
    elif data["domain_info"].get("revenue", "").startswith("$50M"):
        score = "Medium"

    return json.dumps({"lead_score": score})

# Set up a Mapping of the above functions - so it can be called

In [ ]:
# Dictionary of Function Names mapped to Keys (Keys will be used as reference as Tools)
AVAILABLE_FUNCTIONS = {
    "get_domain_info"  : get_domain_info,
    "get_user_info"    : get_user_info,
    "calculate_lead_score": calculate_lead_score,
}

# Configuring the Agent's "Menu" (Tool Schema)

The Large Language Model (LLM) cannot see our Python code directly. We must describe our tools to it using a specific JSON format known as a **Schema**.

This schema tells the model:
* **What** the tool does (Description).
* **When** to use it (Context).
* **How** to use it (Parameters/Arguments).

We pass this list to the `tools` parameter in the API call later. It effectively gives the AI a "menu" of actions it can take.

In [7]:
# Provide the details of availabe Tools to our AI, it has a specific format for OpenAI
tools_schema = [

{
"type": "function",
"function":
    {
    "name": "get_domain_info",  # This is the Function name that will be called.
    "description": "Get the Companys inforamation based on its domain name.",
    "parameters":
        {
        "type": "object",
        "properties":
            {
            "domain": # This is the Input details needed for the function to work.
                {
                "type": "string",
                "description": "The company's domain name, e.g., 'google.com'"
                },
            },
        "required": ["domain"], # Mandatory parameter.
        },
    },
},

{
"type": "function",
"function":
    {
    "name": "get_user_info", # This is the Function name that will be called.
    "description": "Get the User info and notes associated with a specific email.",
    "parameters":
        {
        "type": "object",
        "properties":
            {
            "email": # This is the Input details needed for the function to work.
                {
                "type": "string",
                "description": "The full email address of the lead."
                },
            },
        "required": ["email"], # Mandatory parameter.
        },
    },
},

{
"type": "function",
"function":
    {
    "name": "calculate_lead_score", # This is the Function name that will be called.
    "description": "Calculates the Lead score (High/Medium/Low) for a user based on the Domain data and User Data.",
    "parameters":
        {
        "type": "object",
        "properties":
            {
            "data_summary": # This is the Input details needed for the function to work.
                {
                "type": "string",
                "description": "A JSON string containing the combined Domain data and User Info."
                },
            },
        "required": ["data_summary"], # Mandatory parameter.
        },
    },
},

]